Datasets:
* [openai summarize tldr](https://huggingface.co/datasets/CarperAI/openai_summarize_tldr)
* [openai summarize comparisons](https://huggingface.co/datasets/CarperAI/openai_summarize_comparisons)

Models
* [SmolLM2-135M](https://huggingface.co/HuggingFaceTB/SmolLM2-135M)

## Imports

In [4]:
!pip -q install torch transformers datasets rouge-score matplotlib

In [14]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import random
from datasets import load_dataset
from functools import partial
from torch.utils.data import DataLoader


## Starter Code

In [15]:
MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"  # base model for every stage
SFT_DATA = "CarperAI/openai_summarize_tldr"
PREF_DATA = "CarperAI/openai_summarize_comparisons"

MAX_PROMPT_TOKENS = 448  # post + "TL;DR:" longer than this is dropped
MAX_SUMMARY_TOKENS = 48  # summaries longer than this are dropped

SEED = 42
N_SFT_TRAIN, N_SFT_VAL = 20000, 500
N_PREF_TRAIN, N_PREF_VAL = 20000, 1000
N_EVAL_POSTS = 300

In [16]:
def load_tokenizer(name=MODEL_NAME):
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None:  # SmolLM2 has no pad token
        tok.pad_token = tok.eos_token
    return tok


def load_causal_lm(name_or_path=MODEL_NAME, device=None):
    """Plain pretrained causal LM (no heads added)."""
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = AutoModelForCausalLM.from_pretrained(name_or_path)
    return model.to(device)

In [17]:
TLDR = "TL;DR:"

def clean_post(raw_prompt):
    """Keep the post and end it with the fixed TL;DR marker."""
    post = raw_prompt.split(TLDR)[0].strip()
    return post + "\n" + TLDR


def clean_summary(raw):
    """Remove a leading marker and trailing end-of-text token."""
    text = raw.split("<|endoftext|>")[0].strip()
    if text.startswith(TLDR):
        text = text[len(TLDR):].strip()
    return " " + text


def _sample_valid(rows, n, tok, make, seed):
    order = list(range(len(rows)))
    random.Random(seed).shuffle(order)
    kept = []
    for start in range(0, len(order), 2000):
        for i in order[start:start + 2000]:
            item = make(rows[i], tok)
            if item is not None:
                kept.append(item)
                if len(kept) == n:
                    return kept
    return kept


def _make_sft(row, tok):
    prompt, summary = clean_post(row["prompt"]), clean_summary(row["label"])
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    s = tok(summary, add_special_tokens=False)["input_ids"] + [tok.eos_token_id]
    if len(summary.strip()) == 0 or len(p) > MAX_PROMPT_TOKENS or len(s) > MAX_SUMMARY_TOKENS:
        return None
    return {"prompt_ids": p, "summary_ids": s, "prompt": prompt, "summary": summary}


def _make_pref(row, tok):
    prompt = clean_post(row["prompt"])
    ch, rj = clean_summary(row["chosen"]), clean_summary(row["rejected"])
    if ch == rj or not ch.strip() or not rj.strip():
        return None
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    c = tok(ch, add_special_tokens=False)["input_ids"] + [tok.eos_token_id]
    r = tok(rj, add_special_tokens=False)["input_ids"] + [tok.eos_token_id]
    if len(p) > MAX_PROMPT_TOKENS or max(len(c), len(r)) > MAX_SUMMARY_TOKENS:
        return None
    return {"prompt_ids": p, "chosen_ids": c, "rejected_ids": r, "prompt": prompt, "chosen": ch, "rejected": rj}


def load_sft_splits(tok):
    ds = load_dataset(SFT_DATA)
    train = _sample_valid(ds["train"], N_SFT_TRAIN, tok, _make_sft, SEED)
    val = _sample_valid(ds["valid"], N_SFT_VAL, tok, _make_sft, SEED)
    return train, val


def load_preference_splits(tok):
    ds = load_dataset(PREF_DATA)
    train = _sample_valid(ds["train"], N_PREF_TRAIN, tok, _make_pref, SEED)
    val = _sample_valid(ds["valid1"], N_PREF_VAL, tok, _make_pref, SEED)
    return train, val


def load_eval_posts(tok):
    ds = load_dataset(SFT_DATA)
    return _sample_valid(ds["test"], N_EVAL_POSTS, tok, _make_sft, SEED)

In [18]:
def _pad(seqs, pad_id, side):
    length = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), length), pad_id, dtype=torch.long)
    mask = torch.zeros((len(seqs), length), dtype=torch.long)
    for i, sequence in enumerate(seqs):
        tensor = torch.tensor(sequence, dtype=torch.long)
        if side == "right":
            ids[i, :len(sequence)], mask[i, :len(sequence)] = tensor, 1
        else:
            ids[i, length - len(sequence):], mask[i, length - len(sequence):] = tensor, 1
    return ids, mask

def collate_sft(batch, pad_id):
    sequences = [item["prompt_ids"] + item["summary_ids"] for item in batch]
    ids, mask = _pad(sequences, pad_id, "right")
    prompt_lens = torch.tensor([len(item["prompt_ids"]) for item in batch])
    return {"input_ids": ids, "attention_mask": mask, "prompt_lens": prompt_lens}

def collate_pref(batch, pad_id):
    chosen, chosen_mask = _pad([item["prompt_ids"] + item["chosen_ids"] for item in batch], pad_id, "right")
    rejected, rejected_mask = _pad([item["prompt_ids"] + item["rejected_ids"] for item in batch], pad_id, "right")
    prompt_lens = torch.tensor([len(item["prompt_ids"]) for item in batch])
    return {"chosen_ids": chosen, "chosen_mask": chosen_mask, "rejected_ids": rejected, "rejected_mask": rejected_mask, "prompt_lens": prompt_lens}

def collate_prompts(batch, pad_id):
    ids, mask = _pad([item["prompt_ids"] for item in batch], pad_id, "left")
    return {"input_ids": ids, "attention_mask": mask}

In [19]:
tok = load_tokenizer()
sft_train, sft_val = load_sft_splits(tok)
pref_train, pref_val = load_preference_splits(tok)
eval_posts = load_eval_posts(tok)

print("SFT train/val:", len(sft_train), len(sft_val))
print("Pref train/val:", len(pref_train), len(pref_val))
print("Eval posts:", len(eval_posts))

example = sft_train[0]
print("\n--- One SFT example ---")
print(example["prompt"][-300:])
print("SUMMARY:", repr(example["summary"]))

pad = tok.pad_token_id
batch = next(iter(DataLoader(sft_train, 8, shuffle=True, collate_fn=partial(collate_sft, pad_id=pad))))
print("\nSFT batch:", tuple(batch["input_ids"].shape), "prompt_lens", batch["prompt_lens"].tolist())

batch = next(iter(DataLoader(pref_train, 8, shuffle=True, collate_fn=partial(collate_pref, pad_id=pad))))
print("Pref batch:", tuple(batch["chosen_ids"].shape), tuple(batch["rejected_ids"].shape))

batch = next(iter(DataLoader(eval_posts, 8, collate_fn=partial(collate_prompts, pad_id=pad))))
print("Prompt batch:", tuple(batch["input_ids"].shape), "(left-padded)")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

... (21 outputs left out)

Generating test split:   0%|          | 0/83629 [00:00<?, ? examples/s]

Generating valid1 split:   0%|          | 0/33082 [00:00<?, ? examples/s]

Generating valid2 split:   0%|          | 0/50715 [00:00<?, ? examples/s]

SFT train/val: 20000 500
Pref train/val: 20000 1000
Eval posts: 300

--- One SFT example ---
n thirty minutes. My little sister was in a black car with no A/C in mid-70 degree weather for thirty minutes. She's was still alive, thankfully, but redfaced and crying.

"Crud," my innocent mind thought.

Suffice to say that I didn't leave the house for a while, let alone to go to the park.
TL;DR:
SUMMARY: ' Me and my sister were going to the park, put my sister in her carseat, left her there for thirty minutes, mom saved her from my ignorance, got grounded for a month.'

SFT batch: (8, 408) prompt_lens [262, 193, 316, 264, 214, 243, 376, 181]
Pref batch: (8, 475) (8, 465)
Prompt batch: (8, 378) (left-padded)
